## Faturamento Simulado: 
Nova receita total após limitar os descontos superiores ao teto estipulado.

# 📊 Superstore Analysis: A Anatomia do Sangramento Financeiro
### Investigação Analítica de Margem, Políticas de Desconto Destrutivas e Modelagem para Resgate de Lucro

---

## 🎯 Sumário Executivo & Contexto de Negócio

A **Global Superstore** é uma relevante operação de varejo corporativo e de consumo no mercado norte-americano, atuando através de três grandes verticais de produtos:
* 🪑 **Mobiliário (*Furniture*)**: Mesas, cadeiras, estantes e artigos de decoração.
* 📎 **Material de Escritório (*Office Supplies*)**: Papelaria, fichários, etiquetas e suprimentos corporativos.
* 💻 **Tecnologia (*Technology*)**: Acessórios de TI, copiadoras, máquinas e aparelhos telefônicos.

### 🏢 O Dilema da Liderança
À primeira vista, os demonstrativos contábeis tradicionais apresentavam um retrato de saúde comercial: **um faturamento anual consolidado superior a $ 2,29 milhões e um lucro líquido de $ 286 mil** (margem líquida contábil de **12,47%**).

No entanto, a diretoria executiva detectou uma divergência crônica entre o ritmo de expansão do volume transacionado e o fluxo de caixa efetivamente retido pela operação. A suspeita levantada: **um sangramento financeiro invisível**, em que vendas deficitárias mascaradas pelo volume bruto estavam drenando silenciosamente o resultado gerado pelas operações saudáveis.

### 📌 Painel de Descobertas desta Auditoria Analítica
| Dimensão Analítica | Diagnóstico Real (Cenário Histórico) | Cenário com Teto de Desconto (10%) | Impacto Estratégico no Negócio |
| :--- | :--- | :--- | :--- |
| **Faturamento Total** | **$ 2.297.200,86** | **$ 2.686.332,38** | Recuperação de receita erodida (+16,94%) |
| **Lucro Operacional Líquido** | **$ 286.397,02** | **$ 675.528,54** | **+$ 389.131,52 em lucro (+135,87%)** |
| **Margem Líquida Consolidada** | **12,47%** | **25,15%** | Margem operacional mais que dobra (+12,68 p.p.) |
| **Transações Deficitárias** | **1.871 pedidos (18,72%)** | **330 pedidos (3,30%)** | **Redução de 82,36% das vendas no vermelho** |
| **Faturamento Drenado** | **$ 468.707,15 (20,40%)** | Mitigado por governança | > 1 em cada 5 dólares financiava prejuízo |
| **Prejuízo Acumulado Bruto** | **-$ 156.131,29** | Estancado | Evaporação direta de capital contida |


---

> [!IMPORTANT]
> **A Causa-Raiz Comprovada pelos Dados**:
> O sangramento **não é um problema logístico** (a correlação linear entre prazo de envio e descontos é rigorosamente zero: $r = 0,0004$). Trata-se de uma **falha crítica de governança comercial**: descontos acima de **20%** disparam taxas de prejuízo superiores a **93%**, atingindo **100% de perda** em faixas acima de 30%. O volume comercializado vinha sendo artificialmente sustentado por concessões de preços financeiramente destrutivas.

---

## 🗺️ Roteiro da Investigação
1. **Capítulo 1**: Setup Analítico, Ingestão de Dados & Auditoria Estrutural
2. **Capítulo 2**: Diagnóstico da Anomalia — Identificação das Operações Deficitárias
3. **Capítulo 3**: KPIs Globais de Negócio — A Ilusão dos Grandes Agregados
4. **Capítulo 4**: Auditoria do Sangramento Financeiro — Mensuração da Hemorragia
5. **Capítulo 5**: A Dança dos Descontos — Causa-Raiz e o 'Abismo dos 20%'
6. **Capítulo 6**: Mapeamento dos Vilões do Portfólio — Ranking de Subcategorias Críticas (RF-3)
7. **Capítulo 7**: O Plano de Resgate — Simulação de Sensibilidade com Teto de Desconto (RF-4)
8. **Capítulo 8**: Auditoria Logística — Desmistificando a Hipótese de Lead Time
9. **Conclusão & Plano de Ação Estratégico**

---

# 📦 Capítulo 1: Setup Analítico & Ingestão da Base de Dados

Iniciamos carregando o ambiente e garantindo que o dataset bruto (`superstore_sales_dataset.csv`, com 9.994 transações detalhadas) seja ingerido de forma íntegra a partir do diretório de dados configurado no pacote.

In [ ]:
import pandas as pd

from superstore_sales_dataset.config import RAW_DATA_DIR
from superstore_sales_dataset.metrics import EmptyDataFrameError

### 1.1 Ingestão e Inspeção Estrutural do Dataset
Carregamos os dados em memória e validamos a tipagem dos 21 atributos brutos, assegurando a ausência de registros nulos nos campos transacionais e financeiros.

In [ ]:
sales_data = pd.read_csv(
    RAW_DATA_DIR / "superstore_sales_dataset.csv",
    encoding="ISO-8859-1",
)

In [ ]:
sales_data.info()

### 1.2 Radiografia Estatística Preliminar: Primeiros Sinais de Alerta
Ao avaliar as medidas descritivas globais, buscamos assimetrias nas grandezas financeiras e variações atípicas que sinalizem desvios de rentabilidade.

In [ ]:
sales_data.describe(include="all")

---

# 🔍 Capítulo 2: Diagnóstico da Anomalia — A Existência de Pedidos Deficitários

No comércio varejista tradicional, parte-se do pressuposto de que qualquer transação concluída contribui positivamente para a margem da companhia.
No entanto, a tabela descritiva anterior revelou um dado alarmante: o menor valor na coluna `Profit` é de **-$ 6.599,98**!

Abaixo, quantificamos com precisão a magnitude desse fenômeno:
1. Quantas linhas operam com **lucro estritamente negativo** (`Profit < 0`)?
2. Quantas linhas operam no **ponto de equilíbrio exato** (`Profit == 0`)?

In [ ]:
# Verificando valores < 0 na coluna 'Profit'
negative_profit = sales_data[sales_data["Profit"] < 0]
print(f"Total de registros com lucro negativo: {len(negative_profit)}")

In [ ]:
# Verificando valores =0 na coluna 'Profit'
zero_profit = sales_data[sales_data["Profit"] == 0]
print(f"Total de registros com lucro zero: {len(zero_profit)}")

### 2.1 Formalização do Flag de Transação Deficitária
Para viabilizar a auditoria item a item, criamos o atributo booleano `negative_profit_flag`, isolando as linhas que destroem valor contábil.

In [ ]:
# Criando uma nova coluna para indicar se a linha é de prejuízo (Profit < 0)
sales_data = sales_data.assign(negative_profit_flag=sales_data["Profit"] < 0)
sales_data["negative_profit_flag"].value_counts()

---

## 2.2 Engenharia de Atributos para Auditoria de Rentabilidade

Para quantificar a intensidade da perda em cada pedido e uniformizar a análise, estruturamos dois novos indicadores fundamentais de governança:

1. **`profit_margin_ratio`**: Razão de margem individual por linha de item ($\frac{\text{Profit}}{\text{Sales}}$), com salvaguarda matemática para evitar divisões por zero.
2. **`Is Loss Making`**: Indicador booleano padronizado para rastrear pedidos deficitários nas rotinas de governança e no dashboard executivo.

### Profit Margin Ratio (Margem Unitária por Item):

In [ ]:
sales_data["profit_margin_ratio"] = sales_data.apply(
    lambda row: row["Profit"] / row["Sales"] if row["Sales"] != 0 else 0, axis=1
)

sales_data["profit_margin_ratio"].describe()

### 2.3 Is Loss Making (Identificador Formal de Transação Deficitária)
Variável booleana (`True`/`False`) que identifica se o registro gerou prejuízo direto ($\text{Profit} < 0$).

> [!NOTE]
> Observa-se a presença consistente de **1.871 registros no vermelho** (18,72% da base total), confirmando que a anomalia não é pontual nem decorre de erros de digitação isolados: trata-se de um padrão recorrente de operação.

In [ ]:
# Criando uma nova coluna para indicar se a linha é de prejuízo (Profit < 0)
sales_data["Is Loss Making"] = sales_data["Profit"] < 0

sales_data["Is Loss Making"].value_counts()

---

# 🏛️ Capítulo 3: KPIs Globais de Negócio — A Miragem da Saúde Financeira

Antes de adentrar no detalhamento da sangria de margem, estabelecemos os indicadores consolidados macro da companhia.
Estes números representam o panorama tradicional reportado em relatórios executivos de alto nível, os quais sustentavam a falsa percepção de tranquilidade na operação.

### 3.1 Faturamento Total (Receita Bruta Acumulada)
Soma global da coluna `Sales` ($\sum \text{Sales}$), refletindo a escala macro alcançada pela rede.

In [ ]:
total_revenue = sales_data["Sales"].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")

### 3.2 Lucro Operacional Líquido Consolidado
Soma global da coluna `Profit` ($\sum \text{Profit}$), apurando o resultado financeiro líquido após deduções comerciais.

In [ ]:
consolidated_profit = sales_data["Profit"].sum()
print(f"Consolidated Profit: ${consolidated_profit:,.2f}")

### 3.3 Margem Líquida Consolidada (%)
Razão ponderada entre o Lucro Consolidado e a Receita Bruta Total ($\frac{\sum \text{Profit}}{\sum \text{Sales}} \times 100$).

> [!TIP]
> **Rigor Metodológico**: O cálculo é obrigatoriamente ponderado pelas somas agregadas. Calcular a média aritmética simples das margens unitárias incorreria em distorção estatística grave, tratando pedidos de $ 1 igualmente a pedidos de $ 10.000.

In [ ]:
consolidate_net_margin = consolidated_profit / total_revenue
print(f"Consolidated Net Margin: {consolidate_net_margin:.2%}")

### 3.4 Ticket Médio por Pedido Único
Valor médio faturado por ordem de compra ($\frac{\sum \text{Sales}}{\text{Total de Pedidos Únicos}}$). Agrupamos pelo código de pedido único (`Order ID`), abrangendo as 5.009 compras distintas registradas.

In [ ]:
mean_ticket = total_revenue / len(sales_data["Order ID"].unique())
print(f"Mean Ticket: ${mean_ticket:,.2f}")

---

# 🩸 Capítulo 4: Auditoria do Sangramento Financeiro — Mensuração da Hemorragia

Uma margem líquida agregada de **12,47%** parece aceitável para padrões de varejo de grande escala. Porém, grandes médias consolidam desempenhos estelares com falhas catastróficas.

Isolamos agora exclusivamente o subconjunto de transações com `Is Loss Making == True` para mensurar com precisão cirúrgica o tamanho do sangramento:
* *Quanto lucro foi efetivamente drenado?*
* *Qual a proporção da receita total gerada por operações deficitárias?*
* *Qual a frequência relativa dessas operações danosas?*

### 4.1 Impacto Financeiro Acumulado das Operações Deficitárias
Soma do prejuízo absoluto gerado exclusivamente pelas transações onde $\text{Profit} < 0$.

In [ ]:
total_loss_accumulated = sales_data.loc[sales_data["Is Loss Making"], "Profit"].sum()

print(f"Total Loss Acumulated: $ {total_loss_accumulated:,.2f}")

### 4.2 Faturamento Bruto Envolvido em Vendas Deficitárias
Volume de vendas transacionado em operações que não conseguiram cobrir seus custos diretos:

In [ ]:
# Receita proveniente de transações deficitárias
loss_order_impact = sales_data[sales_data["Is Loss Making"] == "True"]["Sales"].sum()

print(f"Total Loss Orders Impact: ${loss_order_impact:,.2f}")

### 4.3 Proporção de Faturamento Drenado por Operações Deficitárias
Participação percentual da receita originada em transações deficitárias sobre o faturamento total da empresa:
$$\text{Percentual Drenado} = \frac{\sum \text{Sales}_{\text{Profit} < 0}}{\sum \text{Sales}_{\text{Total}}} \times 100$$

In [ ]:
# Percentual de faturamento associado às operações deficitárias
loss_orders_percentage = (
    loss_order_impact / total_revenue * 100 if total_revenue != 0 else None
)

if loss_orders_percentage is not None:
    print(f"Percentage of Loss Orders: {loss_orders_percentage:.2f}%")
else:
    print("Percentage of Loss Orders: Undefined (total revenue is zero)")

### 4.4 Taxa Global de Transações Deficitárias (Frequência Relativa de Falha)
Contagem absoluta e proporção percentual de transações no vermelho sobre o universo de 9.994 linhas operadas pela Superstore.

> [!CAUTION]
> **Diagnóstico da Hemorragia Financeira**:
> * **1.871 transações deficitárias** (18,72% de tudo o que a empresa transaciona).
> * **$ 156.131,29 de lucro evaporados**. Se essas operações tivessem sido simplesmente recusadas, o lucro operacional teria saltado de **$ 286k para $ 442k (+54,5%)**!
> * **20,40% de todo o faturamento da empresa ($ 468.707,15)** financia operações que queimam caixa. A equipe comercial vinha batendo metas de volume vendendo com prejuízo.

In [ ]:
total_loss_sales = (
    sales_data[sales_data["Is Loss Making"] == "True"].value_counts().sum()
)
total_sales = len(sales_data)

percetage_loos_sales = total_loss_sales / total_sales * 100

print(f"Total Sales: {total_sales}")
print(f"Total Loss Sales: {total_loss_sales}")
print(f"Percentage Loss Sales: {percetage_loos_sales:.2f}%")

---

# 📉 Capítulo 5: A Dança dos Descontos — Investigação da Causa Raiz

Comprovada a hemorragia, avançamos para a causa estrutural: **o que torna essas vendas deficitárias?**

No varejo, concessões desmedidas de descontos costumam ser a principal alavanca de destruição de rentabilidade. Para auditar a correlação entre a agressividade dos descontos e o colapso da margem, categorizamos os percentuais praticados em 4 faixas de exposição a risco:
* 🟢 **Low (Baixo)**: $[0.00, 0.10]$ — Descontos pontuais e nominais (0% a 10%)
* 🟡 **Medium (Médio)**: $(0.10, 0.20]$ — Descontos promocionais moderados (10% a 20%)
* 🟠 **High (Alto)**: $(0.20, 0.30]$ — Descontos agressivos (20% a 30%)
* 🔴 **Very High (Crítico)**: $(0.30, 1.00]$ — Concessões extraordinárias (acima de 30%)

In [ ]:
sales_data["Discount_Category"] = pd.cut(
    sales_data["Discount"],
    bins=[-0.01, 0.1, 0.2, 0.3, 1.0],
    labels=["Low", "Medium", "High", "Very High"],
    include_lowest=True,
)

### 5.1 Matriz de Agregação por Categoria e Nível de Desconto
Agrupamos o volume de vendas, o lucro líquido apurado, a contagem de registros com prejuízo e o total de pedidos transacionados em cada cluster.

In [ ]:
sales_data_grup = (
    sales_data.groupby([sales_data["Category"], sales_data["Discount_Category"]])
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        negative_profit_flag=("negative_profit_flag", "sum"),
        total_orders=("Order ID", "count"),
    )
    .reset_index()
)

sales_data_grup.head()

### 5.2 O Ponto de Inflexão da Margem Líquida Pós-Agregação
Calculamos a margem líquida real de cada cluster ($\frac{\text{Profit}}{\text{Sales}} \times 100$) para evidenciar onde a rentabilidade se transforma em prejuízo líquido.

In [ ]:
# Margem Líquida
sales_data_grup["Liquid_Margin_%"] = (
    sales_data_grup["Profit"] / sales_data_grup["Sales"]
) * 100

sales_data_grup[["Category", "Discount_Category", "Sales", "Profit", "Liquid_Margin_%"]]

### 5.3 Taxa de Pedidos Deficitários por Faixa (Probabilidade Empírica de Prejuízo)
Apuramos a probabilidade de um pedido gerar prejuízo quando concedido determinado nível de desconto:
$$\text{Deficit Order Rate \%} = \frac{\text{negative\_profit\_flag}}{\text{total\_orders}} \times 100$$

In [ ]:
# Prejuízo
sales_data_grup["Deficit_Order_Rate_%"] = (
    sales_data_grup["negative_profit_flag"] / sales_data_grup["total_orders"]
) * 100

sales_data_grup[
    [
        "Category",
        "Discount_Category",
        "negative_profit_flag",
        "total_orders",
        "Deficit_Order_Rate_%",
    ]
]

### 5.4 O 'Abismo dos 20%': Evidência Estatística da Inviabilidade Comercial
Filtramos especificamente as faixas de desconto superiores a 20% (`High` e `Very High`) para comprovar a destruição sistemática de margem.

> [!WARNING]
> **A Revelação do Abismo dos 20%:**
> * Na faixa **High (20% a 30%)** em Furniture, **93,24%** dos pedidos dão prejuízo líquido!
> * Na faixa **Very High (>30%)** em Furniture e Office Supplies, **a taxa de perda é de 100,00%**! Não existe um único pedido lucrativo.
> * Em Technology, descontos >30% produzem **84,34%** de pedidos no vermelho.
> 
> **Descontos superiores a 20% ultrapassam a margem de contribuição dos produtos, transformando cada venda concluída em geração garantida de prejuízo.**

In [ ]:
# Filter the DataFrame to include only rows where 'Deficit_Order_Rate_%' is > 0.2

above_20_loss = sales_data_grup[
    sales_data_grup["Discount_Category"].isin(["High", "Very High"])
]

above_20_loss[
    [
        "Category",
        "Discount_Category",
        "negative_profit_flag",
        "total_orders",
        "Deficit_Order_Rate_%",
    ]
]

---

# 🎯 Capítulo 6: O Mapeamento dos Vilões — Ranking de Subcategorias Críticas (RF-3)

A elasticidade e a margem bruta variam substancialmente entre subcategorias de produtos. Enquanto algumas suportam descontos pontuais, outras colapsam rapidamente.

Abaixo, executamos a função de governança `filter_worst_category` para identificar as subcategorias com os piores resultados acumulados de lucro líquido e maior incidência de pedidos no vermelho.

In [ ]:
def filter_worst_category(df: pd.DataFrame) -> pd.DataFrame:
    """Return the subcategories with the lowest total profit."""
    if df.empty:
        raise EmptyDataFrameError()

    worst = (
        df.groupby("Sub-Category")
        .agg(
            Total_Sales=("Sales", "sum"),
            Total_Profit=("Profit", "sum"),
            Negative_Orders=("negative_profit_flag", "sum"),
            Total_Orders=("Order ID", "count"),
        )
        .reset_index()
    )

    worst["Profit_Margin_%"] = worst["Total_Profit"] / worst["Total_Sales"] * 100

    return worst.sort_values("Total_Profit", ascending=True).reset_index(drop=True)


# Execução no notebook
table_rf3 = filter_worst_category(sales_data)
table_rf3


# Execução no notebook
table_rf3 = filter_worst_category(sales_data)
table_rf3

---

# Capítulo 7: Simulador de Sensibilidade com Teto de Desconto (RF-4)

Diante do diagnóstico inequívoco de que descontos excessivos drenam o lucro da empresa, formulamos a proposta estratégica:
**O que aconteceria com os resultados da empresa se estabelecêssemos um teto máximo inegociável de desconto ($d_{\text{teto}}$)?**

### Metodologia da Modelagem Financeira (`simulation_test`):
1. **Reconstrução do Preço Bruto de Tabela**: Reconstruímos o preço original sem o desconto histórico:
   $$P_{\text{bruto}} = \frac{\text{Sales}}{1 - \text{Discount}}$$
2. **Isolamento da Estrutura de Custos**: Deduzimos os custos operacionais históricos associados à mercadoria:
   $$\text{Custo} = \text{Sales} - \text{Profit}$$
3. **Aplicação do Teto Simulado**: Limitamos o desconto histórico ao teto permitido ($d_{\text{novo}} = \min(\text{Discount}, d_{\text{teto}})$).
4. **Recálculo do Faturamento e Lucro**: Apuramos a nova receita e novo lucro gerados sob a nova política comercial.

In [ ]:
def simulation_test(df: pd.DataFrame, roof: float) -> dict[str, float]:
    df_sim = df.copy()

    preco_bruto = df_sim["Sales"] / (1.0 - df_sim["Discount"])
    custo = df_sim["Sales"] - df_sim["Profit"]

    new_discount = df_sim["Discount"].clip(upper=roof)

    df_sim["Sales_simulated"] = preco_bruto * (1.0 - new_discount)
    df_sim["Profit_simulated"] = df_sim["Sales_simulated"] - custo

    original_deficit = (df_sim["Profit"] < 0).sum()
    simulated_deficit = (df_sim["Profit_simulated"] < 0).sum()

    reducao_pct = ((original_deficit - simulated_deficit) / original_deficit) * 100

    return {
        "teto_desconto": roof,
        "faturamento_simulado": df_sim["Sales_simulated"].sum(),
        "lucro_simulado": df_sim["Profit_simulated"].sum(),
        "margem_liquida_simulada_%": (
            df_sim["Profit_simulated"].sum() / df_sim["Sales_simulated"].sum()
        )
        * 100,
        "reducao_pedidos_deficitarios_%": reducao_pct,
        "pedidos_deficitarios_restantes": simulated_deficit,
    }

### 7.1 Cenário Simulado: Governança Rigorosa com Teto de 10% (roof = 0.10)
Testamos o impacto da aplicação de um teto de desconto de 10% sobre todas as transações históricas da empresa.

In [ ]:
table_simulated = pd.DataFrame([simulation_test(sales_data, roof=0.10)])

table_simulated

### 7.2 Recuperação Financeira Líquida (Ganho Incremental de Lucro)
Calculamos a diferença monetária direta resgatada pela eliminação das perdas excessivas:
$$\text{Recuperação Financeira} = \text{Lucro}_{\text{Simulado}} - \text{Lucro}_{\text{Original}}$$

In [ ]:
total_profit = table_rf3["Total_Profit"].sum()
simulated_profit = table_simulated["lucro_simulado"]
financial_recover = simulated_profit.iloc[0] - total_profit
print(f"Simulated financial recover: {financial_recover}")

### 7.3 Validação do KPI de Sucesso do Projeto
Avaliamos se a redução alcançada de transações deficitárias cumpre a meta mínima estabelecida no escopo do projeto (atingir pelo menos **80% de redução** nas operações no vermelho).

In [ ]:
print(
    f"Taxa de Redução de Transações com margem negativa (%): "
    f"{table_simulated['reducao_pedidos_deficitarios_%'].iloc[0]:.2f}"
)

---

# apítulo 8: Auditoria Logística — Desmistificando a Hipótese de Lead Time

Durante a condução do projeto, o time comercial apresentou uma justificativa recorrente:
> *"Os clientes recebem mercadorias com atraso devido à cadeia de suprimentos, e somos forçados a conceder descontos expressivos como compensação comercial."*

Para testar empiricamente essa hipótese, calculamos o **Lead Time** operacional (dias decorridos entre `Order Date` e `Ship Date`) e avaliamos sua correlação matemática com o nível de `Discount` concedido.

### 8.1 Cálculo do Lead Time Operacional (Dias de Trânsito)

In [ ]:
sales_data["Order Date"] = pd.to_datetime(sales_data["Order Date"])
sales_data["Ship Date"] = pd.to_datetime(sales_data["Ship Date"])

sales_data["Lead Time"] = (sales_data["Ship Date"] - sales_data["Order Date"]).dt.days

### 8.2 Correlação Linear Paramétrica (Pearson): Lead Time vs. Desconto

In [ ]:
lead_time_corr = sales_data["Lead Time"].corr(sales_data["Discount"])

print(f"Correlação entre Lead Time e Discount: {lead_time_corr:.4f}")

### 8.3 Correlação Monotônica Não-Paramétrica (Spearman): Lead Time vs. Desconto
Calculamos o coeficiente de Spearman para assegurar que eventuais relações não-lineares ou outliers não mascarem dependências operacionais.

In [ ]:
sales_data[["Lead Time", "Discount"]].corr(method="spearman")

---

# 🏁 Conclusão Executiva & Plano de Ação Estratégico

### 📌 O Veredito dos Dados
1. **A Logística está Totalmente Exonerada**:
   Com correlações nulas ($r = 0,0004$ via Pearson e $\rho = -0,0144$ via Spearman), **não há qualquer fundamento estatístico na alegação de que atrasos de entrega justificam concessões de descontos**. A desculpa operacional foi categoricamente refutada.
2. **A Causa é 100% Comercial e de Governança**:
   A Superstore opera sob um regime permissivo de descontos desprovido de travas automáticas. A equipe comercial priorizou volume de vendas e faturamento bruto às custas da margem operacional da empresa.
3. **O 'Abismo dos 20%'**:
   Concessões de desconto acima de 20% levam a taxas de prejuízo entre 84% e 100%, gerando um ralo financeiro de **$ 156.131,29**.
4. **O Potencial do Teto de Desconto**:
   Limitar os descontos a 10% permite **resgatar $ 389.131,52 em lucro líquido**, dobrar a margem líquida para **25,15%** e extinguir **82,36% das vendas no vermelho**, superando o KPI estipulado para a iniciativa.

---

## 🚀 Roadmap Estratégico para a Liderança (Plano de Ação)

### 1. Governança Imediata de Preços (Curto Prazo - 0 a 30 dias)
* **Travamento no Sistema (Hard Cap)**: Implementar regra sistêmica no ERP/PDV limitando descontos comerciais automáticos ao teto de **10%**.
* **Alçada Diretiva Excepcional**: Descontos entre 10% e 20% passam a depender de aprovação formal de gerência; descontos superiores a **20% ficam terminantemente proibidos**.

### 2. Saneamento do Portfólio Crítico (Médio Prazo - 30 a 60 dias)
* **Revisão Emergencial de Tables e Bookcases**: Renegociar tabela de custos com fabricantes ou reajustar o preço-base de catálogo para restaurar margem mínima positiva nessas duas linhas deficitárias.
* **Blindagem de Binders**: Proibir descontos superiores a 15% em fichários para estancar os 613 pedidos deficitários sem prejudicar o alto giro da categoria.

### 3. Alinhamento de Incentivos da Força de Vendas (Médio/Longo Prazo - 60 a 90 dias)
* **Comissionamento por Margem de Contribuição**: Migrar o modelo de bonificação comercial de *faturamento bruto* para *lucro operacional gerado*. Transações com margem negativa não devem pontuar e devem acarretar desconto proporcional no bônus.

### 4. Monitoramento Contínuo em Tempo Real
* **Operacionalização do Dashboard Streamlit**: Adoção do painel corporativo desenvolvido para monitoramento em tempo real dos pedidos, auditoria de margens e simulação interativa de sensibilidade.